# 03 · HTTP API 직접 호출

라이브러리 없이 `httpx`로 `POST /api/chat`을 부른다. 스트리밍 응답은 **줄마다 JSON 하나**(NDJSON)다. 앞의 두 방식이 안에서 하는 일이 이것이다.

선행: Ollama가 켜져 있고 `qwen3:8b`가 받아져 있어야 한다(00-01).

In [1]:
import json
import time
import httpx

BASE = "http://localhost:11434"

MODEL = "qwen3:8b"
OPTIONS = {"num_ctx": 4096, "temperature": 0.7}   # 세 노트북이 같은 값을 쓴다
THINK = False   # qwen3 사고 과정(<think>) 끄기 — 화면 처리는 Phase 3 전에 정한다
MESSAGES = [
    {"role": "system", "content": "너는 간결하게 한국어로 답하는 도우미다."},
    {"role": "user", "content": "스트리밍 응답이 왜 체감 속도를 높이는지 세 문장으로 설명해줘."},
]

def payload(messages, stream):
    return {"model": MODEL, "messages": messages, "stream": stream, "think": THINK, "options": OPTIONS}

## 1. 비스트리밍
`stream: false`면 JSON 하나가 돌아온다. 한글이 깨지지 않도록 본문은 UTF-8로 읽는다(`r.json()`이 처리).

In [2]:
r = httpx.post(f"{BASE}/api/chat", json=payload(MESSAGES, False), timeout=120)
r.raise_for_status()
d = r.json()
print(d['message']['content'])
print('eval_count =', d['eval_count'], '| eval_duration(ns) =', d['eval_duration'])

스트리밍 응답은 데이터를 실시간으로 전송해 주요 내용을 먼저 보여줘 체감 속도를 높인다.  
중요한 정보가 먼저 도달해 사용자가 빠르게 반응할 수 있도록 돕는다.  
결과적으로 응답이 더 빠르게 느껴지며 사용자 경험을 개선한다.
eval_count = 90 | eval_duration(ns) = 2012964000


## 2. 스트리밍의 날것
응답 줄을 그대로 몇 개 본다. 마지막 줄에만 `done: true`와 통계가 있다.

In [3]:
with httpx.stream("POST", f"{BASE}/api/chat", json=payload(MESSAGES, True), timeout=120) as r:
    r.raise_for_status()
    for i, line in enumerate(r.iter_lines()):
        if i < 3:
            print(line)
        else:
            break

{"model":"qwen3:8b","created_at":"2026-10-04T09:43:12.4981974Z","message":{"role":"assistant","content":"스트"},"done":false}
{"model":"qwen3:8b","created_at":"2026-10-04T09:43:12.5198989Z","message":{"role":"assistant","content":"리"},"done":false}
{"model":"qwen3:8b","created_at":"2026-10-04T09:43:12.5413345Z","message":{"role":"assistant","content":"밍"},"done":false}


## 3. 스트리밍 + tok/s
줄을 하나씩 `json.loads`하고 `message.content`를 이어 붙인다. tok/s 계산은 앞의 두 노트북과 같다.

> 참고: 「첫 토큰까지」 시간은 모델이 메모리에 없으면 적재 시간이 포함된다. 이미 올라가 있으면 0.1초 안팎이다.

In [4]:
t0 = time.perf_counter()
first = None
chunks = 0
last = None
with httpx.stream("POST", f"{BASE}/api/chat", json=payload(MESSAGES, True), timeout=120) as r:
    r.raise_for_status()
    for line in r.iter_lines():
        if not line:
            continue
        d = json.loads(line)
        piece = d["message"]["content"]
        if piece:
            if first is None:
                first = time.perf_counter() - t0
            chunks += 1
            print(piece, end="", flush=True)
        last = d
print()
tps = last["eval_count"] / (last["eval_duration"] / 1e9)
print(f"청크 {chunks}개 | 첫 토큰까지 {first:.2f}s | 생성 {last['eval_count']}토큰 | {tps:.1f} tok/s")

스트

리

밍

 응

답

...

껴

진

다

.


청크 71개 | 첫 토큰까지 2.09s | 생성 78토큰 | 44.3 tok/s


## 4. 멀티턴
메시지 배열을 직접 관리하는 것은 세 방식이 같다.

In [5]:
history = list(MESSAGES)
reply = ""
with httpx.stream("POST", f"{BASE}/api/chat", json=payload(history, True), timeout=120) as r:
    for line in r.iter_lines():
        if line:
            reply += json.loads(line)["message"]["content"]
history.append({"role": "assistant", "content": reply})
history.append({"role": "user", "content": "방금 답을 한 문장으로 줄여줘."})

with httpx.stream("POST", f"{BASE}/api/chat", json=payload(history, True), timeout=120) as r:
    for line in r.iter_lines():
        if line:
            print(json.loads(line)["message"]["content"], end="", flush=True)
print()
print("메시지 수:", len(history))

스트

리

밍

 응

답

...

을

 제공

한다

.


메시지 수: 4
